# BitterTruth-AI — ARC-AGI-3 Competition Submission

**Architecture**: Solver-free cognitive pipeline — OBSERVE → CLASSIFY → EXTRACT_GOAL → MAP_EFFECTS → PLAN → EXECUTE → VERIFY

**Key innovations**:
- H53: Win-state goal templates (learn goals from prior sessions)
- H55: Self-toggle rule extrapolation (plan without visiting every cell)
- H56: Rule-based strategy unlock (escape explore-forever deadlock)
- H51: Autonomous spatial navigation + dead reckoning for movement games
- H47: Score-correlated goal discovery

**No solver data required** — the system discovers mechanics autonomously.

In [ ]:
# ── Environment setup ─────────────────────────────────────────────
import os, sys, time, json, logging

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'

# Suppress verbose logging — keep output clean
logging.basicConfig(level=logging.WARNING)
for noisy in ['arc_agi', 'arcengine', 'engines', 'rungs', 'cognitive_loop']:
    logging.getLogger(noisy).setLevel(logging.ERROR)

# Kaggle: system code is in /kaggle/input/bittertruth-ai/
# Local dev: use current directory
KAGGLE = os.path.exists('/kaggle')
if KAGGLE:
    CODE_DIR = '/kaggle/input/bittertruth-ai'
    ENVS_DIR = '/kaggle/input/arc-agi-3-envs/environment_files'
    KNOWLEDGE_DIR = '/kaggle/input/bittertruth-knowledge'
    sys.path.insert(0, CODE_DIR)
else:
    CODE_DIR = os.path.dirname(os.path.abspath('.'))
    ENVS_DIR = 'environment_files'
    KNOWLEDGE_DIR = 'competition_knowledge'

print(f'Running on: {"Kaggle" if KAGGLE else "Local"}')
print(f'Code:  {CODE_DIR}')
print(f'Games: {ENVS_DIR}')

In [ ]:
# ── SDK + cognitive system imports ────────────────────────────────
from arc_agi import Arcade, OperationMode
from arcengine import GameAction, GameState

from engines.cognition.causal_map import CausalMap

print('Imports OK')

In [ ]:
# ── Initialize arcade in OFFLINE mode ────────────────────────────
# OFFLINE = reads from local environment_files/, no API calls needed.
# This satisfies the Kaggle "no internet" constraint.
arcade = Arcade(
    operation_mode=OperationMode.OFFLINE,
    arc_api_key='',          # Not needed offline
    environments_dir=ENVS_DIR,
)

games = arcade.get_environments()
print(f'Available games: {len(games)}')
for g in games:
    print(f'  {g.game_id}  tags={g.tags}')

In [ ]:
# ── Load pre-built knowledge ──────────────────────────────────────
# During training (evolution runs), the cognitive agent accumulated
# effects, rules, color cycles, and win-state templates.
# These are exported to JSON and bundled as a Kaggle Dataset.
# On competition day, we load them to warm-start each game.

BUNDLED_KNOWLEDGE = {}  # game_id -> knowledge dict

if os.path.exists(KNOWLEDGE_DIR):
    for fname in os.listdir(KNOWLEDGE_DIR):
        if fname.endswith('.json'):
            game_id = fname.replace('.json', '')
            try:
                with open(os.path.join(KNOWLEDGE_DIR, fname)) as f:
                    BUNDLED_KNOWLEDGE[game_id] = json.load(f)
                print(f'  Loaded knowledge: {game_id}')
            except Exception as e:
                print(f'  Warning: could not load {fname}: {e}')
else:
    print('No bundled knowledge directory found — starting cold')

print(f'Knowledge loaded for {len(BUNDLED_KNOWLEDGE)} games')

In [ ]:
# ── Core agent: play one game session ────────────────────────────

def play_game(game_id: str, scorecard_id: str, knowledge: dict = None,
              max_attempts: int = 3, verbose: bool = True,
              knowledge_export_dir: str = None) -> dict:
    """
    Play one game with the cognitive agent.

    Strategy:
      - Attempt 1: Replay known winning sequences (if knowledge available)
      - Attempt 2: Cognitive play with pre-loaded effects/rules
      - Attempt 3: Pure autonomous play (no prior knowledge)

    Returns dict with game_id, score, levels, actions, elapsed.
    """
    results = []
    best_loop = None

    for attempt in range(1, max_attempts + 1):
        t_start = time.time()
        try:
            env = arcade.make(game_id, scorecard_id=scorecard_id, seed=attempt - 1)
            if env is None:
                print(f'  [{game_id}] Could not create environment')
                break

            obs = env.reset()
            available_actions = obs.available_actions
            win_levels = obs.win_levels
            game_type = game_id[:4]

            if verbose:
                is_movement = any(a in available_actions for a in (1, 2, 3, 4))
                is_click = any(a in available_actions for a in (5, 6, 7))
                print(f'  [{game_id}] Attempt {attempt}: actions={available_actions}, '
                      f'win_levels={win_levels}, '
                      f'type={"hybrid" if is_movement and is_click else "move" if is_movement else "click"}')

            # ── Attempt 1: Replay known sequences ────────────────
            if attempt == 1 and knowledge and knowledge.get('winning_sequences'):
                score, levels, actions = _replay_sequences(
                    env, knowledge['winning_sequences'], win_levels, verbose)
                if score > 0:
                    results.append({'attempt': attempt, 'score': score,
                                    'levels': levels, 'actions': actions,
                                    'elapsed': time.time() - t_start})
                    if verbose:
                        print(f'    -> Replay: {levels}/{win_levels} levels, score={score:.3f}')
                    continue  # Try cognitive play next for potentially higher score

            # ── Attempt 2+: Cognitive play ────────────────────────
            score, levels, actions, loop = _cognitive_play(
                env, game_id, game_type, available_actions, win_levels,
                knowledge=knowledge if attempt <= 2 else None,
                verbose=verbose,
            )
            results.append({'attempt': attempt, 'score': score,
                            'levels': levels, 'actions': actions,
                            'elapsed': time.time() - t_start})
            if verbose:
                print(f'    -> Cognitive: {levels}/{win_levels} levels, score={score:.3f}')

            # Keep the loop with highest score for knowledge export
            if best_loop is None or score > (results[-2]['score'] if len(results) > 1 else -1):
                best_loop = loop

            if score >= 1.0:
                break  # Perfect score

        except Exception as e:
            print(f'  [{game_id}] Attempt {attempt} error: {e}')
            import traceback
            traceback.print_exc()

    if not results:
        return {'game_id': game_id, 'score': 0.0, 'levels': 0, 'actions': 0, 'elapsed': 0}

    best = max(results, key=lambda r: r['score'])

    # ── Export learned knowledge ──────────────────────────────────
    if knowledge_export_dir and best_loop and best_loop._causal_map:
        try:
            _export_knowledge(best_loop._causal_map, game_id, knowledge_export_dir)
        except Exception as e:
            if verbose:
                print(f'    Knowledge export failed: {e}')

    return {'game_id': game_id, **best}


print('play_game() defined')

In [ ]:
# ── Replay helper ─────────────────────────────────────────────────

def _replay_sequences(env, sequences_by_level: dict, win_levels: int,
                      verbose: bool) -> tuple:
    """
    Replay pre-recorded winning sequences level by level.
    sequences_by_level: {level_str: [action_entry, ...]}
      where action_entry is int, str "ACTION6", or dict {'action': int, 'data': {...}}
    Returns (score, levels_completed, actions_taken).
    """
    actions_taken = 0
    obs = getattr(env, 'observation_space', None) or env.reset()

    for level_num in range(1, win_levels + 1):
        seq = sequences_by_level.get(str(level_num), [])
        if not seq:
            break

        for seq_entry in seq:
            # Parse entry — support multiple formats
            if isinstance(seq_entry, dict):
                action_num = seq_entry.get('action', seq_entry.get('action_num', 6))
                action_data = seq_entry.get('data', seq_entry.get('action_data', None))
            elif isinstance(seq_entry, int):
                action_num = seq_entry
                action_data = None
            elif isinstance(seq_entry, str) and seq_entry.startswith('ACTION'):
                action_num = int(seq_entry.replace('ACTION', ''))
                action_data = None
            else:
                action_num = int(seq_entry) if seq_entry else 6
                action_data = None

            action = getattr(GameAction, f'ACTION{action_num}', GameAction.ACTION6)
            try:
                obs = env.step(action, data=action_data)
                actions_taken += 1
                if obs.state == GameState.WIN:
                    levels_completed = getattr(obs, 'levels_completed', win_levels)
                    score = levels_completed / win_levels if win_levels > 0 else 1.0
                    return score, levels_completed, actions_taken
                if obs.state == GameState.GAME_OVER:
                    levels_completed = getattr(obs, 'levels_completed', 0) or 0
                    score = levels_completed / win_levels if win_levels > 0 else 0.0
                    return score, levels_completed, actions_taken
            except Exception:
                break

    levels_completed = getattr(obs, 'levels_completed', 0) or 0
    score = levels_completed / win_levels if win_levels > 0 else 0.0
    return score, levels_completed, actions_taken


print('_replay_sequences() defined')

In [ ]:
# ── Cognitive play helper ─────────────────────────────────────────

def _cognitive_play(env, game_id: str, game_type: str, available_actions: list,
                    win_levels: int, knowledge: dict = None,
                    verbose: bool = True):
    """
    Run the full cognitive pipeline on one game session.
    Uses CognitiveLoop directly (no DB, no EvolutionRunner).
    Returns (score, levels_completed, actions_taken, loop).
    Caller can inspect loop._causal_map for knowledge export.
    """
    import numpy as np
    from cognitive_loop import CognitiveLoop

    # Initial observation
    obs = getattr(env, 'observation_space', None) or env.reset()
    max_actions = getattr(obs, 'max_actions', 500) or 500

    # ── Initialize cognitive loop (lightweight — no DB, no rung system) ──
    loop = CognitiveLoop(
        decision_system=None,
        context_builder=None,
        db=None,
        verbose=False,
    )
    loop.start_game(game_id, available_actions, max_actions)

    # ── Warm-start with bundled knowledge (after start_game creates _causal_map) ──
    if knowledge and loop._causal_map:
        _inject_knowledge(loop._causal_map, knowledge)

    actions_taken = 0
    prev_levels = 0
    prev_score = 0.0

    while actions_taken < max_actions:
        frame = getattr(obs, 'frame', None)
        current_available = getattr(obs, 'available_actions', None) or available_actions

        # ── PERCEIVE + THINK + PLAN ──
        try:
            action_num, action_data, cf = loop.cycle(
                frame=frame,
                obs=obs,
                available_actions=current_available,
            )
        except Exception as e:
            if verbose:
                print(f'    cycle() error: {e}')
            import random
            action_num = random.choice(current_available)
            action_data = None

        # Validate action is in allowed set
        if action_num not in current_available:
            import random
            action_num = random.choice(current_available)
            action_data = None

        action = getattr(GameAction, f'ACTION{action_num}', GameAction.ACTION1)

        # ── EXECUTE ──
        try:
            new_obs = env.step(action, data=action_data)
        except Exception as e:
            if verbose:
                print(f'    step() error: {e}')
            break

        if new_obs is None:
            break

        actions_taken += 1

        # ── Compute feedback signals ──
        new_frame = getattr(new_obs, 'frame', None)

        # Frame change: compare pixel arrays (5% threshold)
        frame_changed = False
        if frame is not None and new_frame is not None:
            try:
                fa = np.asarray(frame, dtype=np.float32)
                fb = np.asarray(new_frame, dtype=np.float32)
                if fa.shape == fb.shape:
                    diff_frac = np.mean(np.abs(fa - fb)) / max(fa.max(), 1.0)
                    frame_changed = diff_frac > 0.02
                else:
                    frame_changed = True
            except Exception:
                frame_changed = True

        current_levels = getattr(new_obs, 'levels_completed', 0) or 0
        level_up = current_levels > prev_levels
        current_score = current_levels / win_levels if win_levels > 0 else 0.0
        score_delta = current_score - prev_score

        # ── VERIFY — feed result back to cognitive loop ──
        try:
            loop.record_result(
                post_frame=new_frame,
                frame_changed=frame_changed,
                score_delta=score_delta,
                level_changed=level_up,
                new_level=current_levels + 1 if level_up else 0,
                new_score=current_score,
            )
        except Exception:
            pass

        obs = new_obs
        prev_levels = current_levels
        prev_score = current_score

        # Stop on terminal states
        state = getattr(obs, 'state', None)
        if state in (GameState.WIN, GameState.GAME_OVER):
            break

    final_levels = getattr(obs, 'levels_completed', 0) or 0
    final_score = final_levels / win_levels if win_levels > 0 else 0.0
    return final_score, final_levels, actions_taken, loop


print('_cognitive_play() defined')

In [ ]:
# ── Knowledge injection helper ────────────────────────────────────

def _inject_knowledge(causal_map: 'CausalMap', knowledge: dict):
    """
    Warm-start a CausalMap from bundled JSON knowledge.
    Injects effects, rules, color cycles, win templates, and goal cells.
    """
    from engines.cognition.causal_map import CausalRule, TileEffect

    # Load tile effects
    for pos_key, effect_data in knowledge.get('causal_map', {}).items():
        try:
            parts = pos_key.strip('()').split(',')
            pos = (int(parts[0].strip()), int(parts[1].strip()))

            if isinstance(effect_data, dict) and 'transitions' in effect_data:
                # Rich format from _export_knowledge
                transitions = {}
                for tpos_key, trans_list in effect_data['transitions'].items():
                    tparts = tpos_key.strip('()').split(',')
                    tpos = (int(tparts[0].strip()), int(tparts[1].strip()))
                    transitions[tpos] = [tuple(t) for t in trans_list]
                affected = [tuple(a) for a in effect_data.get('affected', [])]
                te = TileEffect(
                    position=pos,
                    affected=affected,
                    color_transitions=transitions,
                    observation_count=effect_data.get('obs', 1),
                    confidence=effect_data.get('conf', 0.5),
                )
                causal_map._effects[pos] = te
            else:
                # Minimal format — just mark as explored
                pass

            causal_map._explored.add(pos)
            causal_map._all_positions.add(pos)
        except Exception:
            pass

    # Load rules
    for rule_data in knowledge.get('rules', []):
        try:
            causal_map._rules.append(CausalRule(
                rule_type=rule_data['type'],
                description=rule_data.get('desc', ''),
                evidence_count=rule_data.get('evidence', 1),
                confidence=rule_data.get('confidence', 0.5),
            ))
        except Exception:
            pass

    # Load color cycles
    for pos_key, cycle in knowledge.get('color_cycles', {}).items():
        try:
            parts = pos_key.strip('()').split(',')
            pos = (int(parts[0].strip()), int(parts[1].strip()))
            if isinstance(cycle, list):
                causal_map._color_cycles[pos] = cycle
        except Exception:
            pass

    # Load win templates (H53)
    for lvl_str, cells_dict in knowledge.get('win_templates', {}).items():
        try:
            lvl = int(lvl_str)
            cells = {}
            for pos_key, color in cells_dict.items():
                parts = pos_key.strip('()').split(',')
                cells[(int(parts[0].strip()), int(parts[1].strip()))] = int(color)
            if cells:
                causal_map._win_templates[lvl] = cells
        except Exception:
            pass

    # Load goal cells (if win template not available, use saved goal directly)
    goal_cells_data = knowledge.get('goal_cells', {})
    if goal_cells_data and not causal_map._win_templates:
        try:
            for pos_key, color in goal_cells_data.items():
                parts = pos_key.strip('()').split(',')
                pos = (int(parts[0].strip()), int(parts[1].strip()))
                causal_map._goal_cells[pos] = int(color)
            causal_map._goal_source = knowledge.get('goal_source', 'injected')
        except Exception:
            pass

    # Apply L1 win template as goal immediately
    causal_map.apply_win_template(1)


print('_inject_knowledge() defined')

In [ ]:
# ── Knowledge export helper ───────────────────────────────────────

def _export_knowledge(causal_map: 'CausalMap', game_id: str, output_dir: str):
    """
    Serialize learned CausalMap state to JSON for next-run warm-start.
    Exports: tile effects, rules, color cycles, win templates.
    """
    # Tile effects: {pos -> TileEffect}
    effects = {}
    for pos, te in getattr(causal_map, '_effects', {}).items():
        key = f'({pos[0]},{pos[1]})'
        transitions = {}
        for tpos, trans_list in te.color_transitions.items():
            transitions[f'({tpos[0]},{tpos[1]})'] = list(trans_list)
        effects[key] = {
            'affected': list(te.affected),
            'transitions': transitions,
            'obs': te.observation_count,
            'conf': round(te.confidence, 3),
        }

    # Rules
    rules = []
    for r in getattr(causal_map, '_rules', []):
        rules.append({
            'type': r.rule_type,
            'desc': getattr(r, 'description', ''),
            'evidence': getattr(r, 'evidence_count', 1),
            'confidence': round(r.confidence, 3),
        })

    # Color cycles
    color_cycles = {}
    for pos, cycle in getattr(causal_map, '_color_cycles', {}).items():
        color_cycles[f'({pos[0]},{pos[1]})'] = list(cycle)

    # Win templates (H53)
    win_templates = {}
    for lvl, cells in getattr(causal_map, '_win_templates', {}).items():
        win_templates[str(lvl)] = {
            f'({p[0]},{p[1]})': color for p, color in cells.items()
        }

    # Goal cells (for next session's warm-start)
    goal_cells = {}
    for pos, color in getattr(causal_map, '_goal_cells', {}).items():
        goal_cells[f'({pos[0]},{pos[1]})'] = color

    out = {
        'game_id': game_id,
        'causal_map': effects,
        'rules': rules,
        'color_cycles': color_cycles,
        'win_templates': win_templates,
        'goal_cells': goal_cells,
        'goal_source': getattr(causal_map, '_goal_source', ''),
    }

    os.makedirs(output_dir, exist_ok=True)
    path = os.path.join(output_dir, f'{game_id}.json')
    with open(path, 'w') as f:
        json.dump(out, f, indent=2)


print('_export_knowledge() defined')

In [ ]:
# ── Main competition loop ─────────────────────────────────────────
# Opens ONE scorecard, plays ALL games, closes it.
# Exports learned knowledge to /kaggle/working/ after each game.

T_START = time.time()
T_LIMIT_HOURS = 5.5  # Leave 30 min buffer for Kaggle overhead
T_LIMIT_SECS = T_LIMIT_HOURS * 3600

KNOWLEDGE_EXPORT_DIR = '/kaggle/working/knowledge_export' if KAGGLE else 'competition_knowledge_export'
os.makedirs(KNOWLEDGE_EXPORT_DIR, exist_ok=True)

scorecard_id = arcade.create_scorecard(
    source_url='https://github.com/BitterTruth-AI',
    tags=['bittertruth', 'cognitive', 'solver-free'],
)
print(f'Scorecard: {scorecard_id}')
print(f'Playing {len(games)} games with {T_LIMIT_HOURS}h budget')
print(f'Time per game budget: {T_LIMIT_SECS / max(len(games),1) / 60:.1f} min')
print(f'Knowledge export: {KNOWLEDGE_EXPORT_DIR}')
print()

all_results = []

for i, game_info in enumerate(games):
    elapsed = time.time() - T_START
    remaining = T_LIMIT_SECS - elapsed
    games_left = len(games) - i
    time_per_game = remaining / max(games_left, 1)

    if remaining < 60:
        print(f'TIME LIMIT approaching — skipping remaining {games_left} games')
        break

    print(f'[{i+1}/{len(games)}] {game_info.game_id} '
          f'(budget: {time_per_game/60:.1f}min, elapsed: {elapsed/60:.1f}min total)')

    knowledge = BUNDLED_KNOWLEDGE.get(game_info.game_id) or \
                BUNDLED_KNOWLEDGE.get(game_info.game_id[:4], {})

    result = play_game(
        game_info.game_id,
        scorecard_id=scorecard_id,
        knowledge=knowledge,
        max_attempts=2,
        verbose=True,
        knowledge_export_dir=KNOWLEDGE_EXPORT_DIR,
    )
    all_results.append(result)
    print(f'  RESULT: score={result["score"]:.3f}, '
          f'levels={result.get("levels", 0)}, '
          f'time={result.get("elapsed", 0):.1f}s')
    print()

print('=' * 60)
print(f'DONE. Total time: {(time.time()-T_START)/60:.1f} min')
print(f'Games played: {len(all_results)}/{len(games)}')
avg_score = sum(r["score"] for r in all_results) / max(len(all_results), 1)
print(f'Average score: {avg_score:.4f}')
print()

In [ ]:
# ── Final scorecard + submission ──────────────────────────────────
final_scorecard = arcade.close_scorecard(scorecard_id)

if final_scorecard:
    print(f'Final scorecard score: {final_scorecard.score:.4f}')
    print(f'Environments completed: '
          f'{final_scorecard.total_environments_completed}/{final_scorecard.total_environments}')
    print()
    print('Per-game scores:')
    for env_score in final_scorecard.environments:
        print(f'  {env_score.id}: {env_score.score:.3f}')
else:
    print('Warning: scorecard not available (offline mode — expected)')
    print('Results summary:')
    for r in sorted(all_results, key=lambda x: x['score'], reverse=True):
        print(f'  {r["game_id"]}: score={r["score"]:.3f}')